In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
from fuzzywuzzy import process
from constants import DATA_FILENAME, INITIAL_FILE, SILVER_FILE, KENPOM_FILE

pd.set_option('expand_frame_repr', False)
pd.set_option('display.max_columns', 0)  # Display any number of columns
pd.set_option('display.max_rows', 68)  # Display any number of rows

In [3]:
FIRST_FOUR_LOSERS = {
    "mens": [
        "San Diego St.",
        "Saint Francis (PA)",
        "American",
        "Texas"
    ],
    "womens": [
        "Princeton",
        "Washington",
        "High Point",
        "UC San Diego"
    ],
}

In [4]:
GENDERS = ["mens", "womens"]
results = {}
for gender in GENDERS:
    initial = pd.read_excel(INITIAL_FILE.format(gender=gender))
    silver = pd.read_excel(SILVER_FILE.format(gender=gender), skiprows=15).rename(columns={"Team": "team_name", "Composite": "team_rating"})
    espn = initial.merge(silver[["team_name", "team_rating"]], on="team_name")
    simple = espn[['team_region', 'team_seed', 'team_name', 'team_rating']].rename(columns={"team_seed": "Seed"})
    simple["Seed"] = simple["Seed"].replace("\D", "", regex=True)
    simple = simple.loc[lambda x: ~x["team_name"].isin(FIRST_FOUR_LOSERS[gender])]
    results[gender] = simple

In [5]:
[r.head() for r in results.values()]

[  team_region Seed    team_name  team_rating
 0       South    1       Auburn  2074.434605
 1       South   16  Alabama St.  1355.205356
 3       South    8   Louisville  1907.835539
 4       South    9    Creighton  1853.688991
 5       South    5     Michigan  1891.817819,
   team_region Seed     team_name  team_rating
 0   Spokane 1    1          UCLA  2309.785067
 2   Spokane 1   16      Southern  1428.419275
 3   Spokane 1    8      Richmond  1940.269539
 4   Spokane 1    9  Georgia Tech  1933.461755
 5   Spokane 1    5      Ole Miss  2063.569948]

In [6]:
[r.shape for r in results.values()]

[(64, 4), (64, 4)]

In [7]:
for gender, r in results.items():
    r.to_csv(DATA_FILENAME.format(gender=gender), index=False)

# Add KenPom to Mens

In [8]:
kenpom = pd.read_csv(KENPOM_FILE.format(gender="mens"))
team_info = pd.read_csv(DATA_FILENAME.format(gender="mens"))

In [9]:
kenpom.tail()

,Rank,Seed,Team,Conference,Wins,Losses,AdjustEM,AdjustO,AdjustO Rank,AdjustD,AdjustD Rank,AdjustT,AdjustT Rank,Luck,Luck Rank,SOS Pyth,SOS Pyth Rank,SOS OppO,SOS OppO Rank,SOS OppD,SOS OppD Rank,NCSOS Pyth,NCSOS Pyth Rank
59,161,14,Montana,BSky,25,9,0.24,110.7,98,110.5,252,67.1,194,0.186,1,-2.36,205,106.4,180,108.8,239,8.56,28
60,180,16,Norfolk St.,MEAC,24,10,-1.50,107.2,166,108.7,213,66.4,227,0.056,58,-7.18,325,103.1,306,110.3,336,3.09,90
61,216,16,SIUE,OVC,22,11,-4.25,102.0,255,106.2,160,65.9,261,0.052,67,-9.01,357,101.4,349,110.4,345,-3.37,276
62,238,16,Mount St. Mary's,MAAC,23,12,-6.03,101.1,279,107.1,183,67.5,165,0.147,2,-7.16,323,102.0,333,109.1,267,-1.16,211
63,273,16,Alabama St.,SWAC,20,15,-9.25,101.4,273,110.7,256,67.8,144,0.061,51,-8.82,355,100.7,359,109.5,298,6.21,45


In [10]:
# Match Kenpom and ESPN team names
kenpom['team_name'] = np.nan
choices = team_info['team_name'].tolist()  # Get all of the ESPN team names
for index, row in kenpom.iterrows():
    team = row['Team']
    match = process.extractOne(team, choices)  # extract the best match for every kenpom name
    if match[1] == 100:  # it's a perfect match
        espn_name = team_info[team_info['team_name'] == match[0]]['team_name'].values[0]
        kenpom.loc[index, 'team_name'] = espn_name
        choices.remove(match[0])  # remove that team as a possible choice

# Need to do this first, otherwise the >80 match ¯\_(ツ)_/¯
# kenpom.loc[kenpom['Team'] == "St. John's", 'team_id'] = 2599
# choices.remove("St. John's (NY)")
# kenpom.loc[kenpom['Team'] == "Ohio St.", 'team_id'] = 194
# choices.remove("Ohio State")
# kenpom.loc[kenpom['Team'] == "Saint Peter's", 'team_id'] = 2612
# choices.remove("St. Peter's")
# kenpom.loc[kenpom['Team'] == "N.C. State", 'team_id'] = 152
# choices.remove("North Carolina State")

scores = {}
for index, row in kenpom[kenpom['team_name'].isna()].iterrows():
    team = row['Team']
    match = process.extractOne(team, choices)
    espn_name = team_info[team_info['team_name'] == match[0]]['team_name'].values[0]
    scores[team] = match, espn_name
    if match[1] > 80:
        kenpom.loc[index, 'team_name'] = espn_name
        choices.remove(match[0])  # remove that team as a possible choice

missing_kenpom = kenpom.loc[kenpom['team_name'].isna(), 'Team'].tolist()
missing_espn = team_info.loc[team_info['team_name'].isin(choices)][['team_name']]

print("Missing Kenpom Teams: {}".format(missing_kenpom))
print("Matching ESPN Teams:\n {}".format(missing_espn))

Missing Kenpom Teams: ['Mississippi', 'Connecticut', 'SIUE']
Matching ESPN Teams:
            team_name
8           Ole Miss
18             UConn
33  SIU Edwardsville


In [11]:
scores

{"Saint Mary's": (("Saint Mary's (CA)", 95), "Saint Mary's (CA)"),
 'Mississippi': (('Ole Miss', 42), 'Ole Miss'),
 'Connecticut': (('UConn', 72), 'UConn'),
 'Nebraska Omaha': (('Omaha', 90), 'Omaha'),
 'SIUE': (('SIU Edwardsville', 68), 'SIU Edwardsville'),
 "Mount St. Mary's": (("Mount St. Mary's (MD)", 95), "Mount St. Mary's (MD)")}

In [12]:
# Need to manually fix the crappy ones
# (kenpom_name, espn_name)
manual_matches = [
    ('Mississippi', 'Ole Miss'),
    ('Connecticut', 'UConn'),
    ('SIUE', 'SIU Edwardsville'),
]
for team, team_id in manual_matches:
    kenpom.loc[kenpom['Team'] == team, 'team_name'] = team_id

merged = kenpom.merge(team_info, on=['team_name', 'Seed'])

simple = merged[['team_region', 'Seed', 'team_name', 'AdjustEM', 'AdjustT', 'team_rating']]

In [13]:
simple.head()

,team_region,Seed,team_name,AdjustEM,AdjustT,team_rating
0,East,1,Duke,38.25,65.8,2122.116374
1,West,1,Florida,36.19,69.6,2094.260328
2,Midwest,1,Houston,35.41,61.5,2107.215646
3,South,1,Auburn,35.11,67.6,2074.434605
4,Midwest,2,Tennessee,31.13,63.7,2007.081523


In [14]:
simple.tail()

,team_region,Seed,team_name,AdjustEM,AdjustT,team_rating
59,East,14,Montana,0.24,67.1,1556.058620
60,West,16,Norfolk St.,-1.50,66.4,1498.150394
61,Midwest,16,SIU Edwardsville,-4.25,65.9,1442.857915
62,East,16,Mount St. Mary's (MD),-6.03,67.5,1445.747651
63,South,16,Alabama St.,-9.25,67.8,1355.205356


In [15]:
simple.shape

(64, 6)

In [16]:
simple.to_csv(DATA_FILENAME.format(gender="mens"), index=False)

## OLD

In [27]:
# espn = pd.read_csv(ESPN_FILE)
# GENDERS = ["mens", "womens"]
# forecast_dates = espn.groupby(
#     ["gender", "forecast_date", "results_to"]
# ).filter(lambda x: len(x) == 64).groupby("gender")["forecast_date"].unique().to_dict()
# print(forecast_dates)
# results = {}
# for gender in GENDERS:
#     forecast_date = forecast_dates[gender]
#     assert len(forecast_date) == 1
#     espn_slice = (
#         (espn['gender'] == gender) &
#         (espn['forecast_date'] == forecast_date[0]) &
#         (espn["team_alive"] == 1) 
#     )
#     # For backdated runs
#     # espn_slice = (espn['gender'] == 'mens') & (espn['forecast_date'] == "2023-03-15") & (espn["team_alive"] == 1)
#     simple = espn[espn_slice][['team_region', 'team_seed', 'team_name', 'team_id', 'team_rating']].rename(columns={"team_seed": "Seed"})
#     simple["Seed"] = simple["Seed"].str.replace("\D", "", regex=True)
#     results[gender] = simple